# Video FaceSwap on Colab GPU

Headless InsightFace `inswapper_128` pipeline (same model family Rope uses) + Gradio UI.
Your weak local GPU is never used — everything runs here on the Colab T4.

**Setup:** Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.

> Only swap faces you have consent to use.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print(sys.version)
try:
    import torch
    print('torch CUDA available:', torch.cuda.is_available())
except Exception as e:
    print('torch check skipped:', e)

In [ ]:
# 1) Get the app code. Set REPO_URL after you push this folder to GitHub.
#    (Or upload the files manually — then skip the git clone.)
REPO_URL = 'https://github.com/divyanshmusicc/faceswap.git'

%cd /content
import os
if os.path.exists('/content/faceswap/app/swapper.py'):
    print('app code already present')
elif 'YOU' not in REPO_URL:
    !git clone $REPO_URL
else:
    print('⚠️ Set REPO_URL above, or use Files → Upload to place the faceswap folder at /content/faceswap')

%cd /content/faceswap
!pip install -r requirements-colab.txt
print('deps installed')

In [ ]:
# 2) Download models (~550MB total). Skips files that already exist.
!mkdir -p /root/.insightface/models
!test -f /root/.insightface/models/inswapper_128.onnx || wget -O /root/.insightface/models/inswapper_128.onnx https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx
!test -d /root/.insightface/models/buffalo_l || (cd /tmp && wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip && unzip -q -o buffalo_l.zip -d /root/.insightface/models/buffalo_l && rm buffalo_l.zip)
!ls -lh /root/.insightface/models/inswapper_128.onnx
!ls /root/.insightface/models/buffalo_l

In [ ]:
# 3) Optional: mount Drive so uploads / outputs survive Colab timeouts.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 4) Smoke test: CUDA provider visible + models load.
%cd /content/faceswap
import onnxruntime as ort
print('providers:', ort.get_available_providers())
from app.swapper import ensure_models, has_gpu
ensure_models()
print('models OK, has_gpu =', has_gpu())

In [ ]:
# 5) Launch the app. Open the gradio.live link it prints.
#    Keep this cell running while you use the app.
%cd /content/faceswap
from app.gradio_app import create_demo
demo = create_demo()
demo.queue().launch(share=True)